# Estudo de Caso 9.3 — Caldeira Industrial para Geração de Vapor

**Capítulo Associado:** Capítulo 9 — Trocadores de Calor  
**Nível:** Graduação/Pós-Graduação  
**Tempo estimado:** 90 minutos  
**Autor:** Jader Lugon Junior

---

## 📋 Resumo

Neste estudo de caso, dimensionamos uma **caldeira flamotubular** para uma pequena indústria alimentícia. A caldeira converte energia química (combustão de gás natural) em energia térmica, promovendo a geração de vapor para processos industriais.

**Objetivos:**
1. Calcular a carga térmica requerida (vaporização + pré-aquecimento)
2. Verificar o balanço de energia no lado dos gases
3. Estimar os coeficientes convectivos (gases e ebulição)
4. Calcular o coeficiente global de transferência de calor
5. Determinar a LMTD e a área de troca necessária
6. Redimensionar o trocador para atender à carga térmica

---

## 🎯 Objetivos de Aprendizagem

- Aplicar balanço de energia em trocador com mudança de fase
- Estimar coeficientes convectivos para ebulição e gases
- Calcular coeficiente global considerando incrustação
- Dimensionar trocador casco-tubo com geometria complexa
- Analisar sensibilidade a variações operacionais
- Introduzir problemas inversos para calibração de parâmetros

---

## 🔧 Requisitos

- Bibliotecas: `numpy`, `matplotlib`, `scipy`
- Conhecimento prévio: LMTD, método ε-NTU, correlações de Nusselt (Cap. 9)
- Conceitos de ebulição nucleada e propriedades dos gases de combustão

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Ambiente configurado com sucesso!')

---

## 🏭 Seção 1: Contexto Industrial

Uma **caldeira** é um trocador de calor projetado para converter energia química (combustão) em energia térmica, promovendo a geração de vapor. O projeto é baseado na transferência de calor entre os gases quentes e um fluido (água) para mudança de estado de líquido para vapor.

### Tipos de Caldeiras

Entre os esquemas de funcionamento, é possível adotar:
- **Aquatubular (água nos tubos):** Água circula nos tubos e gases quentes passam pelo casco. Usado em caldeiras de grande porte.
- **Flamotubular (gases nos tubos):** Gases quentes passam nos tubos e água está no casco. Usado em caldeiras de pequeno e médio porte.

Neste estudo de caso, consideramos uma **caldeira flamotubular** para uma pequena indústria alimentícia.

### Normas de Segurança

No Brasil, o projeto e operação de caldeiras deve obedecer à **NR13** do Ministério do Trabalho. Requisitos essenciais:

1. **Válvula de segurança:** Para evitar sobrepressão
2. **Manômetro:** Instrumento que indique a pressão do vapor
3. **Sistema de alimentação de água:** Injetor ou sistema independente do principal
4. **Sistema de drenagem rápida:** Para emergências
5. **Controle automático de nível:** Para evitar queima seca

---

## 📊 Seção 2: Descrição do Problema

### Dados de Projeto

| Parâmetro | Valor | Unidade |
|-----------|-------|----------|
| **Fluido quente (gases)** | | |
| Vazão mássica | $\dot{m}_h = 1,2$ | kg/s |
| Temperatura de entrada | $T_{h,in} = 900$ | °C |
| Temperatura de saída (estimada) | $T_{h,out} \approx 250$ | °C |
| Calor específico | $c_{p,h} = 1100$ | J/(kg·K) |
| Viscosidade dinâmica | $\mu_h = 4,5 \times 10^{-5}$ | Pa·s |
| Condutividade térmica | $k_h = 0,065$ | W/(m·K) |
| Número de Prandtl | $\mathrm{Pr}_h = 0,72$ | adimensional |
| **Fluido frio (água/vapor)** | | |
| Pressão de operação | $p = 10$ | bar |
| Temperatura de saturação | $T_{sat} = 179,9$ | °C |
| Vazão de vapor desejada | $\dot{m}_v = 0,3$ | kg/s |
| Calor latente de vaporização | $h_{fg} = 2015$ | kJ/kg |
| Temperatura da água de alimentação | $T_{c,in} = 80$ | °C |
| **Geometria proposta** | | |
| Tipo | Casco-tubo, 1 casco, 2 passes | |
| Tubos de aço carbono | $D_i = 25$ mm, $D_o = 30$ mm | |
| Condutividade do tubo | $k_{tubo} = 45$ | W/(m·K) |
| Número de tubos | $N_t = 40$ | |
| Comprimento por tubo | $L = 3,0$ | m |
| Arranjo | Triangular, passo $S_T = 40$ mm | |
| **Condições operacionais** | | |
| Fator de incrustação interno (gases) | $R_{f,i} = 0,0004$ | m²·K/W |
| Fator de incrustação externo (água/vapor) | $R_{f,o} = 0,0002$ | m²·K/W |
| Rendimento da combustão | $\eta_{comb} = 0,92$ | adimensional |

**Objetivos:**
1. Verificar se a área proposta é suficiente para gerar $\dot{m}_v = 0,3$ kg/s
2. Calcular $U$ e $\Delta T_{lm}$
3. Estimar temperaturas de saída reais

---

## 🔥 Seção 3: Carga Térmica Requerida

### Passo 1: Calor para Vaporização

O calor necessário para vaporizar a água a 10 bar é:

$$\dot{Q}_{vap} = \dot{m}_v \cdot h_{fg} = 0,3 \times 2015 \times 10^3 = 604.500 \text{ W}$$

### Passo 2: Calor para Pré-aquecimento

O calor necessário para pré-aquecer a água de alimentação de 80°C até a temperatura de saturação (179,9°C) é:

$$\dot{Q}_{pre} = \dot{m}_v \cdot c_{p,agua} \cdot (T_{sat} - T_{c,in})$$

$$\dot{Q}_{pre} = 0,3 \times 4180 \times (179,9 - 80) = 125.400 \text{ W}$$

### Passo 3: Carga Térmica Total

$$\dot{Q}_{req} = \dot{Q}_{vap} + \dot{Q}_{pre} = 604.500 + 125.400 = 729.900 \text{ W} \approx 730 \text{ kW}$$

**Interpretação:** A caldeira deve fornecer **730 kW** de calor para gerar 0,3 kg/s de vapor a 10 bar, considerando o pré-aquecimento da água de alimentação.

In [ ]:
# ============================================================
# CARGA TÉRMICA REQUERIDA
# ============================================================

# Dados do fluido frio (água/vapor)
m_v = 0.3  # kg/s (vazão de vapor)
h_fg = 2015e3  # J/kg (calor latente)
cp_agua = 4180  # J/(kg·K)
T_sat = 179.9  # °C (temperatura de saturação a 10 bar)
T_c_in = 80  # °C (temperatura da água de alimentação)

# Cálculos
Q_vap = m_v * h_fg  # W
Q_pre = m_v * cp_agua * (T_sat - T_c_in)  # W
Q_req = Q_vap + Q_pre  # W

print('=' * 60)
print('CARGA TÉRMICA REQUERIDA')
print('=' * 60)
print(f'\nVazão de vapor: ṁ_v = {m_v} kg/s')
print(f'Calor latente: h_fg = {h_fg/1000:.0f} kJ/kg')
print(f'Temperatura de saturação: T_sat = {T_sat}°C')
print(f'Temperatura da água: T_c,in = {T_c_in}°C')
print(f'\nCalor para vaporização: Q_vap = {Q_vap/1000:.1f} kW')
print(f'Calor para pré-aquecimento: Q_pre = {Q_pre/1000:.1f} kW')
print(f'\nCarga térmica total: Q_req = {Q_req/1000:.1f} kW')

# Gráfico de contribuição
fig, ax = plt.subplots(figsize=(8, 5))

labels = ['Vaporização', 'Pré-aquecimento']
sizes = [Q_vap/1000, Q_pre/1000]
colors = ['red', 'orange']

ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Distribuição da Carga Térmica', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n📊 Interpretação:')
print(f'  • Vaporização: {Q_vap/Q_req*100:.0f}% da carga total')
print(f'  • Pré-aquecimento: {Q_pre/Q_req*100:.0f}% da carga total')
print(f'  • A vaporização domina, como esperado em caldeiras')

---

## 🔥 Seção 4: Verificação do Balanço nos Gases

### Passo 4: Calor Disponível dos Gases

O calor disponível dos gases de combustão, considerando o rendimento da combustão, é:

$$\dot{Q}_{disp} = \eta_{comb} \cdot \dot{m}_h \cdot c_{p,h} \cdot (T_{h,in} - T_{h,out,est})$$

Substituindo:

$$\dot{Q}_{disp} = 0,92 \times 1,2 \times 1100 \times (900 - 250) = 792.720 \text{ W}$$

### Passo 5: Verificação

Como $\dot{Q}_{disp} > \dot{Q}_{req}$ (792,7 kW > 730 kW), há **margem térmica suficiente**. Os gases podem fornecer o calor necessário para gerar o vapor desejado.

**Margem de segurança:**

$$\text{Margem} = \frac{\dot{Q}_{disp} - \dot{Q}_{req}}{\dot{Q}_{req}} \times 100 = \frac{792,7 - 730}{730} \times 100 \approx 8,6\%$$

**Interpretação:** A margem de 8,6% é adequada para compensar perdas térmicas, variações operacionais e incertezas nos parâmetros.

In [ ]:
# ============================================================
# BALANÇO NOS GASES
# ============================================================

# Dados do fluido quente (gases)
m_h = 1.2  # kg/s
cp_h = 1100  # J/(kg·K)
T_h_in = 900  # °C
T_h_out_est = 250  # °C (estimativa)
eta_comb = 0.92  # rendimento da combustão

# Cálculo
Q_disp = eta_comb * m_h * cp_h * (T_h_in - T_h_out_est)  # W

# Margem
margem = (Q_disp - Q_req) / Q_req * 100

print('=' * 60)
print('BALANÇO NOS GASES')
print('=' * 60)
print(f'\nVazão de gases: ṁ_h = {m_h} kg/s')
print(f'Calor específico: c_p,h = {cp_h} J/(kg·K)')
print(f'Temperatura de entrada: T_h,in = {T_h_in}°C')
print(f'Temperatura de saída (estimada): T_h,out = {T_h_out_est}°C')
print(f'Rendimento da combustão: η_comb = {eta_comb*100:.0f}%')
print(f'\nCalor disponível: Q_disp = {Q_disp/1000:.1f} kW')
print(f'Carga requerida: Q_req = {Q_req/1000:.1f} kW')
print(f'\nMargem de segurança: {margem:.1f}%')

# Verificação
if Q_disp > Q_req:
    print(f'\n✅ Margem suficiente! Os gases podem fornecer o calor necessário.')
else:
    print(f'\n❌ Margem INSUFICIENTE! Redimensionar o trocador.')

# Temperatura de saída real (se toda a carga for atendida)
T_h_out_real = T_h_in - Q_req / (eta_comb * m_h * cp_h)

print(f'\n🌡️ Temperatura de saída real dos gases:')
print(f'   T_h,out = {T_h_out_real:.1f}°C')
print(f'   (vs. estimativa de {T_h_out_est}°C)')

---

## 🌡️ Seção 5: Cálculo da LMTD

### Característica Especial: Mudança de Fase

Como a água evapora a **temperatura constante** ($T_{sat} = 179,9°C$), o perfil de temperatura é característico de um fluido em mudança de fase.

### Passo 6: Diferenças de Temperatura nas Extremidades

$$\Delta T_1 = T_{h,in} - T_{sat} = 900 - 179,9 = 720,1 \text{ K}$$

$$\Delta T_2 = T_{h,out,est} - T_{sat} = 250 - 179,9 = 70,1 \text{ K}$$

### Passo 7: LMTD

$$\Delta T_{lm} = \frac{\Delta T_1 - \Delta T_2}{\ln(\Delta T_1 / \Delta T_2)} = \frac{720,1 - 70,1}{\ln(720,1/70,1)} = \frac{650}{2,33} \approx 279 \text{ K}$$

**Interpretação:** A LMTD de 279 K é muito alta, refletindo a grande diferença de temperatura entre os gases quentes e a água em ebulição. Isso é típico de caldeiras.

In [ ]:
# ============================================================
# CÁLCULO DA LMTD
# ============================================================

# Diferenças de temperatura
dT1 = T_h_in - T_sat
dT2 = T_h_out_est - T_sat

# LMTD
LMTD = (dT1 - dT2) / np.log(dT1 / dT2)

print('=' * 60)
print('CÁLCULO DA LMTD')
print('=' * 60)
print(f'\nTemperatura de saturação: T_sat = {T_sat}°C (constante)')
print(f'\nDiferenças de temperatura:')
print(f'  • ΔT₁ = T_h,in - T_sat = {dT1:.1f} K')
print(f'  • ΔT₂ = T_h,out - T_sat = {dT2:.1f} K')
print(f'  • Razão ΔT₁/ΔT₂ = {dT1/dT2:.2f}')
print(f'\nLMTD = {LMTD:.1f} K')

# Gráfico do perfil de temperatura
fig, ax = plt.subplots(figsize=(10, 5))

# Eixo x: posição no trocador (0 a 1)
x = np.linspace(0, 1, 100)

# Água/vapor (temperatura constante)
T_agua_profile = np.full_like(x, T_sat)

# Gases (perfil exponencial decrescente)
T_gases_profile = T_sat + (T_h_in - T_sat) * np.exp(-x * np.log(dT1/dT2))

ax.plot(x, T_gases_profile, 'r-', linewidth=3, label='Gases (resfriando)')
ax.plot(x, T_agua_profile, 'b-', linewidth=3, label='Água (ebulição)')

# Preencher área entre as curvas
ax.fill_between(x, T_agua_profile, T_gases_profile, alpha=0.2, color='orange',
                label=f'LMTD = {LMTD:.0f} K')

# Anotações
ax.annotate(f'ΔT₁ = {dT1:.0f} K', xy=(0, T_h_in), xytext=(0.1, T_h_in - 30),
            fontsize=10, fontweight='bold', color='red')
ax.annotate(f'ΔT₂ = {dT2:.0f} K', xy=(1, T_h_out_est), xytext=(0.8, T_h_out_est - 30),
            fontsize=10, fontweight='bold', color='red')

ax.set_xlabel('Posição no Trocador')
ax.set_ylabel('Temperatura [°C]')
ax.set_title('Perfil de Temperatura na Caldeira', fontweight='bold')
ax.legend(loc='best')
ax.set_xlim(0, 1)
ax.set_ylim(150, 950)
ax.set_xticks([0, 0.5, 1])
ax.set_xticklabels(['Entrada gases', 'Meio', 'Saída gases'])

plt.tight_layout()
plt.show()

---

## 🔧 Seção 6: Estimativa dos Coeficientes Convectivos

### Passo 8: Coeficiente do Lado dos Tubos (Gases)

Para escoamento transversal sobre feixe de tubos, usamos a correlação de Zukauskas:

$$\mathrm{Nu}_D = C \cdot \mathrm{Re}_D^m \cdot \mathrm{Pr}^{0,36}$$

onde $C$ e $m$ dependem do arranjo (triangular ou quadrado) e do número de Reynolds.

**Cálculo do Reynolds:**

$$\mathrm{Re}_D = \frac{\rho_h \cdot u_{max} \cdot D_o}{\mu_h}$$

Para gases a 600°C (temperatura média), $\rho_h \approx 0,35$ kg/m³. A velocidade máxima $u_{max}$ é calculada a partir da área mínima de escoamento cruzado.

Com $\mathrm{Re}_D \approx 8500$ (turbulento), obtemos:

$$\mathrm{Nu}_D = 0,27 \times 8500^{0,63} \times 0,72^{0,36} \approx 62$$

$$h_i = \frac{\mathrm{Nu}_D \cdot k_h}{D_o} = \frac{62 \times 0,065}{0,030} \approx 134 \text{ W/(m}^2\cdot\text{K)}$$

### Passo 9: Coeficiente do Lado do Casco (Ebulição)

Para ebulição nucleada em tubos horizontais, usamos a correlação de Chen ou simplificações:

$$h_o \approx 3000 \text{--} 8000 \text{ W/(m}^2\cdot\text{K)}$$

Adotamos um valor conservador:

$$h_o = 4000 \text{ W/(m}^2\cdot\text{K)}$$

**Interpretação:** O coeficiente de ebulição é muito maior que o coeficiente dos gases, o que é típico em caldeiras. A resistência dominante está no lado dos gases.

In [ ]:
# ============================================================
# COEFICIENTES CONVECTIVOS
# ============================================================

# Dados dos gases
rho_h = 0.35  # kg/m³ (gases a 600°C)
mu_h = 4.5e-5  # Pa·s
k_h = 0.065  # W/(m·K)
Pr_h = 0.72

# Geometria
D_o = 0.030  # m
S_T = 0.040  # m (passo transversal)
N_t = 40  # número de tubos
L_tubo = 3.0  # m

# Cálculo da velocidade máxima
# Área mínima de escoamento cruzado (aproximação)
A_min = N_t * S_T * L_tubo / 2  # divisão por 2 para 2 passes
u_max = m_h / (rho_h * A_min)

# Reynolds
Re_D = rho_h * u_max * D_o / mu_h

# Nusselt (Zukauskas simplificado)
Nu_D = 0.27 * Re_D**0.63 * Pr_h**0.36

# Coeficiente convectivo (gases)
h_i = Nu_D * k_h / D_o

# Coeficiente convectivo (ebulição)
h_o = 4000  # W/(m²·K)

print('=' * 60)
print('COEFICIENTES CONVECTIVOS')
print('=' * 60)
print(f'\nLado dos tubos (gases):')
print(f'  • Velocidade máxima: u_max = {u_max:.2f} m/s')
print(f'  • Reynolds: Re_D = {Re_D:.0f}')
print(f'  • Nusselt: Nu_D = {Nu_D:.1f}')
print(f'  • Coeficiente: h_i = {h_i:.0f} W/(m²·K)')
print(f'\nLado do casco (ebulição):')
print(f'  • Coeficiente: h_o = {h_o:.0f} W/(m²·K)')
print(f'\n📊 Razão h_o/h_i = {h_o/h_i:.0f}')
print(f'   A ebulição é {h_o/h_i:.0f}× mais eficiente que o escoamento de gases')

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 5))

labels = ['Gases (h_i)', 'Ebulição (h_o)']
values = [h_i, h_o]
colors = ['red', 'blue']

ax.bar(labels, values, color=colors, alpha=0.7)
ax.set_ylabel('Coeficiente Convectivo [W/(m²·K)]')
ax.set_title('Comparação dos Coeficientes', fontweight='bold')
ax.set_yscale('log')

# Anotações
for i, v in enumerate(values):
    ax.text(i, v * 1.1, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🔧 Seção 7: Cálculo do Coeficiente Global $U$

### Passo 10: Áreas de Transferência

A área interna (baseada no diâmetro interno) é:

$$A_i = \pi \cdot D_i \cdot L \cdot N_t = \pi \times 0,025 \times 3,0 \times 40 = 9,42 \text{ m}^2$$

A área externa (baseada no diâmetro externo) é:

$$A_o = \pi \cdot D_o \cdot L \cdot N_t = \pi \times 0,030 \times 3,0 \times 40 = 11,31 \text{ m}^2$$

### Passo 11: Resistências Térmicas em Série

O coeficiente global referido à área externa é:

$$\frac{1}{U_o \cdot A_o} = \frac{1}{h_i \cdot A_i} + \frac{R_{f,i}}{A_i} + \frac{\ln(D_o/D_i)}{2\pi \cdot k_{tubo} \cdot L \cdot N_t} + \frac{R_{f,o}}{A_o} + \frac{1}{h_o \cdot A_o}$$

### Passo 12: Cálculo Numérico

Substituindo os valores:

$$\frac{1}{U_o \times 11,31} = \frac{1}{134 \times 9,42} + \frac{0,0004}{9,42} + \frac{\ln(30/25)}{2\pi \times 45 \times 3,0 \times 40} + \frac{0,0002}{11,31} + \frac{1}{4000 \times 11,31}$$

$$\frac{1}{U_o \times 11,31} = 7,92 \times 10^{-4} + 4,25 \times 10^{-5} + 2,16 \times 10^{-6} + 1,77 \times 10^{-5} + 2,21 \times 10^{-5}$$

$$\frac{1}{U_o \times 11,31} = 8,60 \times 10^{-4}$$

$$U_o = \frac{1}{11,31 \times 8,60 \times 10^{-4}} \approx 103 \text{ W/(m}^2\cdot\text{K)}$$

**Interpretação:** O coeficiente global de 103 W/(m²·K) é baixo, refletindo a resistência dominante no lado dos gases ($h_i = 134$ W/(m²·K)). A ebulição tem coeficiente muito alto ($h_o = 4000$ W/(m²·K)), mas isso não compensa a baixa eficiência do lado dos gases.

In [ ]:
# ============================================================
# COEFICIENTE GLOBAL U
# ============================================================

# Dados geométricos
D_i = 0.025  # m
k_tubo = 45  # W/(m·K)

# Fatores de incrustação
R_fi = 0.0004  # m²·K/W
R_fo = 0.0002  # m²·K/W

# Áreas
A_i = np.pi * D_i * L_tubo * N_t
A_o = np.pi * D_o * L_tubo * N_t

# Resistências térmicas
R_conv_i = 1 / (h_i * A_i)
R_fouling_i = R_fi / A_i
R_wall = np.log(D_o / D_i) / (2 * np.pi * k_tubo * L_tubo * N_t)
R_fouling_o = R_fo / A_o
R_conv_o = 1 / (h_o * A_o)

# Resistência total
R_total = R_conv_i + R_fouling_i + R_wall + R_fouling_o + R_conv_o

# Coeficiente global
U_o = 1 / (R_total * A_o)

print('=' * 60)
print('COEFICIENTE GLOBAL U')
print('=' * 60)
print(f'\nÁreas:')
print(f'  • Área interna: A_i = {A_i:.2f} m²')
print(f'  • Área externa: A_o = {A_o:.2f} m²')
print(f'\nResistências térmicas [K/W]:')
print(f'  • Convecção interna (gases): {R_conv_i:.4e} ({R_conv_i/R_total*100:.1f}%)')
print(f'  • Incrustação interna: {R_fouling_i:.4e} ({R_fouling_i/R_total*100:.1f}%)')
print(f'  • Condução na parede: {R_wall:.4e} ({R_wall/R_total*100:.1f}%)')
print(f'  • Incrustação externa: {R_fouling_o:.4e} ({R_fouling_o/R_total*100:.1f}%)')
print(f'  • Convecção externa (ebulição): {R_conv_o:.4e} ({R_conv_o/R_total*100:.1f}%)')
print(f'\nResistência total: R_total = {R_total:.4e} K/W')
print(f'\nCoeficiente global: U_o = {U_o:.0f} W/(m²·K)')

# Gráfico de contribuição
fig, ax = plt.subplots(figsize=(10, 5))

labels = ['Conv. gases', 'Incrust. int.', 'Parede', 'Incrust. ext.', 'Ebulição']
values = [R_conv_i, R_fouling_i, R_wall, R_fouling_o, R_conv_o]
colors = ['red', 'orange', 'gray', 'yellow', 'blue']

ax.bar(labels, [v/R_total*100 for v in values], color=colors, alpha=0.7)
ax.set_ylabel('Contribuição [%]')
ax.set_title('Distribuição das Resistências Térmicas', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n📊 Interpretação:')
print(f'  • Convecção dos gases domina: {R_conv_i/R_total*100:.0f}% da resistência')
print(f'  • Ebulição tem resistência desprezível: {R_conv_o/R_total*100:.1f}%')
print(f'  • Melhorar h_i é a chave para aumentar U')

---

## 📏 Seção 8: Verificação da Área Necessária

### Passo 13: Área Requerida

$$A_{req} = \frac{\dot{Q}_{req}}{U_o \cdot \Delta T_{lm}} = \frac{730.000}{103 \times 279} \approx 25,4 \text{ m}^2$$

### Passo 14: Comparação com Área Proposta

A área proposta (externa) é:

$$A_o = 11,31 \text{ m}^2$$

Como $A_{req} > A_o$ (25,4 m² > 11,3 m²), a **área proposta é insuficiente**!

**Déficit de área:**

$$\text{Déficit} = \frac{A_{req} - A_o}{A_o} \times 100 = \frac{25,4 - 11,3}{11,3} \times 100 \approx 125\%$$

**Conclusão:** A caldeira proposta não consegue gerar os 0,3 kg/s de vapor desejados. É necessário redimensionar o trocador.

In [ ]:
# ============================================================
# VERIFICAÇÃO DA ÁREA
# ============================================================

# Área requerida
A_req = Q_req / (U_o * LMTD)

# Déficit
deficit = (A_req - A_o) / A_o * 100

print('=' * 60)
print('VERIFICAÇÃO DA ÁREA')
print('=' * 60)
print(f'\nÁrea requerida: A_req = {A_req:.1f} m²')
print(f'Área proposta: A_o = {A_o:.1f} m²')
print(f'\nDéficit: {deficit:.0f}%')

# Verificação
if A_req > A_o:
    print(f'\n❌ Área INSUFICIENTE! Redimensionar o trocador.')
else:
    print(f'\n✅ Área suficiente!')

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 5))

labels = ['Requerida', 'Proposta']
values = [A_req, A_o]
colors = ['red', 'green']

ax.bar(labels, values, color=colors, alpha=0.7)
ax.set_ylabel('Área [m²]')
ax.set_title('Comparação de Áreas', fontweight='bold')

# Anotações
for i, v in enumerate(values):
    ax.text(i, v + 0.5, f'{v:.1f} m²', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🔧 Seção 9: Redimensionamento do Trocador

### Opções para Aumentar a Área

1. **Aumentar o número de tubos** ($N_t$)
2. **Aumentar o comprimento dos tubos** ($L$)
3. **Melhorar $h_i$** (ex: turbuladores, aletas nos tubos)

### Passo 15: Aumentar o Número de Tubos

Escolhendo aumentar $N_t$:

$$N_{t,novo} = N_t \times \frac{A_{req}}{A_o} = 40 \times \frac{25,4}{11,3} \approx 90 \text{ tubos}$$

### Passo 16: Recalcular $U_o$ com Nova Geometria

Com $N_t = 90$, as áreas tornam-se:

$$A_i = \pi \times 0,025 \times 3,0 \times 90 = 21,21 \text{ m}^2$$

$$A_o = \pi \times 0,030 \times 3,0 \times 90 = 25,45 \text{ m}^2$$

Recalculando $U_o$ (iterativo, pois $h_i$ pode mudar com a nova geometria):

$$U_o \approx 98 \text{ W/(m}^2\cdot\text{K)}$$

### Passo 17: Nova Área Requerida

$$A_{req,novo} = \frac{730.000}{98 \times 279} \approx 26,7 \text{ m}^2$$

Como $A_o = 25,45$ m² é ligeiramente menor que $A_{req,novo} = 26,7$ m², podemos:
- Aumentar para 95 tubos
- Ou aceitar uma pequena redução na vazão de vapor

**Conclusão:** Com **90-95 tubos**, a caldeira atende à carga térmica de 730 kW.

In [ ]:
# ============================================================
# REDIMENSIONAMENTO
# ============================================================

# Novo número de tubos
N_t_novo = int(N_t * A_req / A_o)

# Novas áreas
A_i_novo = np.pi * D_i * L_tubo * N_t_novo
A_o_novo = np.pi * D_o * L_tubo * N_t_novo

# Recalcular U_o (aproximação)
R_conv_i_novo = 1 / (h_i * A_i_novo)
R_fouling_i_novo = R_fi / A_i_novo
R_wall_novo = np.log(D_o / D_i) / (2 * np.pi * k_tubo * L_tubo * N_t_novo)
R_fouling_o_novo = R_fo / A_o_novo
R_conv_o_novo = 1 / (h_o * A_o_novo)

R_total_novo = R_conv_i_novo + R_fouling_i_novo + R_wall_novo + R_fouling_o_novo + R_conv_o_novo
U_o_novo = 1 / (R_total_novo * A_o_novo)

# Nova área requerida
A_req_novo = Q_req / (U_o_novo * LMTD)

print('=' * 60)
print('REDIMENSIONAMENTO DO TROCADOR')
print('=' * 60)
print(f'\nNovo número de tubos: N_t = {N_t_novo}')
print(f'\nNovas áreas:')
print(f'  • Área interna: A_i = {A_i_novo:.2f} m²')
print(f'  • Área externa: A_o = {A_o_novo:.2f} m²')
print(f'\nNovo coeficiente global: U_o = {U_o_novo:.0f} W/(m²·K)')
print(f'Nova área requerida: A_req = {A_req_novo:.1f} m²')

# Verificação
if A_o_novo >= A_req_novo:
    print(f'\n✅ Área suficiente! Caldeira atende à carga térmica.')
else:
    print(f'\n⚠️ Área ainda insuficiente. Aumentar para {int(N_t_novo * A_req_novo / A_o_novo) + 1} tubos.')

# Gráfico de evolução
fig, ax = plt.subplots(figsize=(10, 5))

labels = ['Original\n(40 tubos)', 'Redimensionado\n(90 tubos)']
areas_propostas = [A_o, A_o_novo]
areas_requeridas = [A_req, A_req_novo]

x = np.arange(len(labels))
width = 0.35

ax.bar(x - width/2, areas_propostas, width, label='Área Proposta', color='green', alpha=0.7)
ax.bar(x + width/2, areas_requeridas, width, label='Área Requerida', color='red', alpha=0.7)

ax.set_ylabel('Área [m²]')
ax.set_title('Evolução do Dimensionamento', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

plt.tight_layout()
plt.show()

print(f'\n📋 Resumo do redimensionamento:')
print(f'  • Aumentar de {N_t} para {N_t_novo} tubos')
print(f'  • Área passa de {A_o:.1f} para {A_o_novo:.1f} m²')
print(f'  • U_o muda de {U_o:.0f} para {U_o_novo:.0f} W/(m²·K)')
print(f'  • Caldeira agora atende à carga de {Q_req/1000:.0f} kW')

---

## 📋 Seção 10: Resumo dos Resultados

| Parâmetro | Valor Original | Valor Redimensionado |
|-----------|---------------|---------------------|
| Número de tubos | 40 | 90 |
| Área interna $A_i$ | 9,42 m² | 21,21 m² |
| Área externa $A_o$ | 11,31 m² | 25,45 m² |
| Coeficiente global $U_o$ | 103 W/(m²·K) | 98 W/(m²·K) |
| LMTD | 279 K | 279 K |
| Área requerida | 25,4 m² | 26,7 m² |
| Carga térmica atendida | Insuficiente | 730 kW ✓ |

---

## 🔍 Seção 11: Análise de Sensibilidade

### Efeito de $h_i$ no Coeficiente Global

Como a convecção dos gases domina a resistência térmica, melhorar $h_i$ é a chave para aumentar $U_o$.

**Cenários:**

| Melhoria em $h_i$ | Novo $h_i$ [W/(m²·K)] | Novo $U_o$ [W/(m²·K)] | Redução de Área |
|-------------------|----------------------|----------------------|----------------|
| Base | 134 | 103 | — |
| +20% | 161 | 115 | ~10% |
| +50% | 201 | 132 | ~22% |
| +100% | 268 | 156 | ~34% |

**Estratégias para melhorar $h_i$:**
1. **Turbuladores:** Insertos que aumentam a turbulência
2. **Aletas internas:** Aumentam a área de transferência
3. **Maior velocidade dos gases:** Aumentam $\mathrm{Re}$ e $\mathrm{Nu}$
4. **Geometria otimizada:** Passo e arranjo dos tubos

### Efeito da Incrustação

A incrustação é crítica em caldeiras, especialmente no lado dos gases (fuligem, cinzas).

**Cenários:**

| Fator de Incrustação $R_{f,i}$ | $U_o$ [W/(m²·K)] | Redução |
|--------------------------------|------------------|--------|
| Limpo ($R_{f,i} = 0$) | 115 | — |
| $R_{f,i} = 0,0004$ (projeto) | 103 | 10% |
| $R_{f,i} = 0,0008$ (2×) | 92 | 20% |
| $R_{f,i} = 0,0012$ (3×) | 83 | 28% |

**Conclusão:** Dobrar a incrustação reduz $U_o$ em 20%, exigindo limpeza periódica.

In [ ]:
# ============================================================
# ANÁLISE DE SENSIBILIDADE
# ============================================================

# Efeito de h_i
h_i_range = np.array([134, 161, 201, 268])
U_o_range = []

for h_i_teste in h_i_range:
    R_conv_i_teste = 1 / (h_i_teste * A_i_novo)
    R_total_teste = R_conv_i_teste + R_fouling_i_novo + R_wall_novo + R_fouling_o_novo + R_conv_o_novo
    U_o_teste = 1 / (R_total_teste * A_o_novo)
    U_o_range.append(U_o_teste)

U_o_range = np.array(U_o_range)

# Efeito de R_fi
R_fi_range = np.array([0, 0.0004, 0.0008, 0.0012])
U_o_Rf_range = []

for R_fi_teste in R_fi_range:
    R_fouling_i_teste = R_fi_teste / A_i_novo
    R_total_teste = R_conv_i_novo + R_fouling_i_teste + R_wall_novo + R_fouling_o_novo + R_conv_o_novo
    U_o_teste = 1 / (R_total_teste * A_o_novo)
    U_o_Rf_range.append(U_o_teste)

U_o_Rf_range = np.array(U_o_Rf_range)

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Efeito de h_i
axes[0].plot(h_i_range, U_o_range, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Coeficiente h_i [W/(m²·K)]')
axes[0].set_ylabel('Coeficiente Global U_o [W/(m²·K)]')
axes[0].set_title('Efeito de h_i no Coeficiente Global', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: Efeito de R_fi
axes[1].plot(R_fi_range * 10000, U_o_Rf_range, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Fator de Incrustação R_fi [×10⁻⁴ m²·K/W]')
axes[1].set_ylabel('Coeficiente Global U_o [W/(m²·K)]')
axes[1].set_title('Efeito da Incrustação', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=' * 60)
print('ANÁLISE DE SENSIBILIDADE')
print('=' * 60)
print(f'\n📊 Efeito de h_i:')
for h_i_val, U_o_val in zip(h_i_range, U_o_range):
    reducao = (1 - U_o_val / U_o_range[0]) * 100
    print(f'  • h_i = {h_i_val:.0f} → U_o = {U_o_val:.0f} W/(m²·K)')

print(f'\n📊 Efeito da incrustação:')
for R_fi_val, U_o_val in zip(R_fi_range, U_o_Rf_range):
    reducao = (1 - U_o_val / U_o_Rf_range[0]) * 100
    print(f'  • R_fi = {R_fi_val:.4f} → U_o = {U_o_val:.0f} W/(m²·K) (redução: {reducao:.0f}%)')

---

## 📝 Seção 12: Exercícios Resolvidos

### Exercícios de Graduação

Os exercícios de graduação foram resolvidos ao longo das seções:

1. **Carga térmica requerida:** 730 kW (vaporização + pré-aquecimento)
2. **Balanço nos gases:** Margem de 8,6% (suficiente)
3. **LMTD:** 279 K
4. **Coeficientes convectivos:** $h_i = 134$ W/(m²·K), $h_o = 4000$ W/(m²·K)
5. **Coeficiente global:** $U_o = 103$ W/(m²·K)
6. **Verificação de área:** Insuficiente (25,4 m² > 11,3 m²)
7. **Redimensionamento:** 90 tubos

---

### Exercícios de Pós-Graduação

#### Exercício 1: Variação de $k(T)$

**Enunciado:** Suponha que a condutividade térmica do alumínio varie linearmente com a temperatura: $k(T) = k_0 [1 + \beta(T - T_\infty)]$. Deduza a equação algébrica não-linear resultante para o nó $i$ e proponha um algoritmo iterativo para resolver o sistema.

**Solução:**

A equação da aleta com $k$ variável é:

$$\frac{d}{dx}\left(k(T) \frac{dT}{dx}\right) - \frac{hP}{A_c}(T - T_\infty) = 0$$

Discretizando por diferenças finitas:

$$\frac{k_{i+1/2}(T_{i+1} - T_i) - k_{i-1/2}(T_i - T_{i-1})}{(\Delta x)^2} - m^2(T_i - T_\infty) = 0$$

onde $k_{i+1/2} = k\left(\frac{T_i + T_{i+1}}{2}\right) = k_0 \left[1 + \beta\left(\frac{T_i + T_{i+1}}{2} - T_\infty\right)\right]$

**Algoritmo iterativo (Picard):**

1. Inicialize $T_i^{(0)}$ com solução para $k$ constante
2. Para $k = 0, 1, 2, \dots$:
   - Calcule $k_{i+1/2}^{(k)}$ usando $T_i^{(k)}$
   - Monte o sistema linear $A^{(k)} T^{(k+1)} = b$
   - Resolva para $T^{(k+1)}$
   - Verifique convergência: $\|T^{(k+1)} - T^{(k)}\| < \text{tol}$

#### Exercício 2: Estimação de $h$ e $k$ por Problema Inverso

**Enunciado:** Utilize o método de Levenberg-Marquardt para calibrar $h$ e $k$ simultaneamente a partir de dados sintéticos de perfil térmico gerados com ruído. Analise a matriz de covariância e discuta a correlação entre os parâmetros estimados.

**Solução:**

**Formulação do problema inverso:**

Minimizar a função custo:

$$S(h,k) = \sum_{i=1}^{N} \left[ T_{\text{calc}}(x_i; h,k) - T_{\text{med}}(x_i) \right]^2$$

**Método de Levenberg-Marquardt:**

$$\Delta \mathbf{P} = \left( \mathbf{J}^T \mathbf{J} + \mu \mathbf{I} \right)^{-1} \mathbf{J}^T \mathbf{r}$$

onde $\mathbf{P} = [h, k]^T$, $\mathbf{J}$ é a matriz Jacobiana (sensibilidades), $\mathbf{r}$ é o vetor de resíduos.

**Projeto sugerido:**

1. Implemente o solver direto 1D para a equação da aleta
2. Gere dados sintéticos com $h=75$ W/(m²·K), $k=205$ W/(m·K), adicionando ruído gaussiano ($\sigma = 1°C$)
3. Implemente o método LM para estimar $h$ e $k$ a partir dos dados sintéticos
4. Analise a matriz de covariância e os intervalos de confiança (95%) das estimativas
5. Discuta a influência do número e posição dos sensores na qualidade da estimação

**Referência:** Lugon Jr. et al. (2008) aplicaram abordagem similar para estimação de coeficientes de dispersão em rios. O código-fonte está disponível em:

https://github.com/JaderLugon/fenomenos-transporte-livro

---

## 💡 Seção 13: Discussão e Conclusões

### Principais Aprendizados

1. **Balanço de energia:** A caldeira requer 730 kW para gerar 0,3 kg/s de vapor, com 82,8% destinado à vaporização e 17,2% ao pré-aquecimento

2. **Resistência dominante:** A convecção dos gases ($h_i = 134$ W/(m²·K)) domina a resistência térmica, enquanto a ebulição ($h_o = 4000$ W/(m²·K)) tem resistência desprezível

3. **Dimensionamento inadequado:** A proposta inicial com 40 tubos era insuficiente, exigindo redimensionamento para 90 tubos

4. **Sensibilidade:** Melhorar $h_i$ em 20% reduz a área necessária em ~10%, enquanto dobrar a incrustação reduz $U_o$ em 20%

5. **Segurança:** A caldeira deve obedecer à NR13, com válvula de segurança, manômetro, controle de nível e sistema de drenagem

### Recomendações Práticas

1. **Projeto conservador:** Dimensionar para $T_{h,out} = 200°C$ (não 250°C), garantindo margem para variações operacionais

2. **Melhoria de $h_i$:** Considerar turbuladores ou aletas internas para aumentar o coeficiente dos gases

3. **Controle de incrustação:** Instalar sopradores de fuligem e programar limpeza periódica dos tubos

4. **Monitoramento contínuo:** Instalar sensores de temperatura, pressão e vazão para detecção precoce de problemas

5. **Conformidade normativa:** Elaborar prontuário da caldeira conforme NR13, com inspeções periódicas

### Aplicações Práticas

Os conceitos deste estudo de caso são aplicáveis a:

- **Indústria alimentícia:** Geração de vapor para cozimento, esterilização e pasteurização
- **Indústria têxtil:** Tingimento e secagem de tecidos
- **Indústria química:** Reações endotérmicas e destilação
- **Hospitais:** Esterilização de equipamentos e lavanderias
- **Edifícios comerciais:** Sistemas de aquecimento e ar condicionado

---

## 🔄 Navegação

- [📚 Voltar ao Capítulo 9](../notebooks/09_Trocadores_Calor.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**Estudo de Caso 9.3**  
Parte da coleção "Fenômenos de Transporte: Fundamentos e Modelagem Computacional"

</div>